# Amparo -- M3 · S08: RAG avanzado

Monta las dos tecnicas de recuperacion avanzada (hybrid search y reranking)
sobre el RAG base. El retrieval como herramienta (tool use) y el agente ReAct
de la S10 estan en `m3_s10_rag_agentico.ipynb`.
Toda la logica vive en `tools/rag/`; aca se clona el repo, se instala el stack
pesado y se corren las fases.

**Las tres configuraciones que se comparan** (lo unico que cambia entre ellas es
el retrieval; mismo prompt, mismo generador, mismo eval set):

| Config | `use_hybrid` | `use_rerank` | Retrieval |
|---|---|---|---|
| **A** | False | False | denso puro (coseno e5) |
| **B** | True | False | denso + BM25, fusion RRF |
| **C** | True | True | hybrid -> cross-encoder reordena |

**Requiere GPU** (T4 o superior): e5, el cross-encoder y Qwen2.5-7B corren aca.

Las decisiones de diseno de cada tecnica estan en `docs/m3_decisiones_rag.md`.

Cada respuesta se emite en formato Ragas (`pipeline.to_eval_record`) con los
flags `used_hybrid`/`used_rerank`, que identifican la configuracion que la
genero.

In [1]:
# Rama del repo a clonar. Cambiala si el codigo del RAG avanzado todavia no esta
# en main (por ejemplo, mientras vive en una rama de PR).
RAMA = "m3.5"

import os

if not os.path.isdir("Amparo"):
    !git clone -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd Amparo && git fetch origin && git checkout {RAMA} && git pull

%cd Amparo

Cloning into 'Amparo'...
remote: Enumerating objects: 1005, done.
remote: Counting objects: 100% (651/651), done.
remote: Compressing objects: 100% (354/354), done.
remote: Total 1005 (delta 410), reused 493 (delta 282), pack-reused 354 (from 1)
Receiving objects: 100% (1005/1005), 4.38 MiB | 23.74 MiB/s, done.
Resolving deltas: 100% (599/599), done.
/content/Amparo


In [2]:
import os, sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

REQUERIDOS = [
    "tools/rag/hybrid.py",
    "tools/rag/rerank.py",
    "tools/rag/tools.py",
    "tools/rag/pipeline.py",
    "data/eval_set.json",
]
faltan = [r for r in REQUERIDOS if not os.path.exists(r)]
if faltan:
    raise SystemExit(
        f"Falta en la rama '{RAMA}': {faltan}\n\n"
        "Este notebook corre contra el repo REMOTO. Si el codigo del RAG avanzado "
        "todavia esta solo en tu maquina, commitealo y pusheralo, o cambia RAMA."
    )
print("Repo OK.")

Repo OK.


In [3]:
# Dependencias livianas del repo (incluye faiss-cpu y rank_bm25, el BM25 de la
# hybrid search).
!pip install -q -r requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 126.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.1 MB/s eta 0:00:00


In [4]:
# Stack pesado: se instala aqui y no en requirements.txt, para no reemplazar el
# build de PyTorch con CUDA que Colab ya trae. sentence-transformers trae el
# cross-encoder del reranking; transformers alcanza para e5 y la generacion;
# peft/bitsandbytes solo si se genera con el adaptador LoRA.
!pip install -q -U transformers sentence-transformers peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 144.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 61.9 MB/s eta 0:00:00


## Fase 1 -- Cargar el indice y construir el BM25

El indice denso (FAISS) se construye en `m3_s07_rag_ingenuo.ipynb` y se guarda en Drive.
Aca se carga ya hecho. El indice BM25 se construye en memoria a partir de la
metadata del store (los mismos chunks, en el mismo orden): es Python puro y
tarda segundos; se construye una vez y se reusa en toda la corrida.

In [5]:
from google.colab import drive
import pathlib, shutil
from tools.rag import config

drive.mount("/content/drive")
origen = pathlib.Path(config.DRIVE_ROOT) / "rag"
config.ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
for nombre in ("rag_index.faiss", "rag_index_metadata.jsonl"):
    shutil.copy(origen / nombre, config.ARTIFACTS_DIR / nombre)
print("Indice copiado de Drive a artifacts/.")

Mounted at /content/drive
Indice copiado de Drive a artifacts/.


In [6]:
from tools.rag import pipeline
from tools.rag.hybrid import BM25Index

store = pipeline.load_index()
bm25 = BM25Index(store.metadata)   # una sola vez; se reusa en cada consulta
print(f"Indice denso: {len(store)} chunks | BM25: {len(bm25)} chunks")

Indice denso: 3425 chunks | BM25: 3425 chunks


## Fase 2 -- Comparacion lado a lado: A vs B vs C sobre el retrieval

Antes de generar, se mira solo lo que recupera cada configuracion. Tres consultas
elegidas para exponer donde cada tecnica aporta:

1. **Termino exacto** (`articulo 64` del CST): donde el denso difumina y BM25
   rescata el chunk correcto -> B/C deberian subirlo.
2. **Coloquial** (lenguaje del usuario, sin terminos tecnicos): donde el denso ya
   es fuerte -> hybrid no debe empeorarlo.
3. **Fuera del corpus**: la valvula de escape debe activarse en las tres.

Solo se recupera, sin generar: es el diagnostico por consulta, mas informativo
que la metrica agregada.

In [7]:
from tools.rag.retrieve import retrieve

SISTEMAS = {
    "A denso": dict(use_hybrid=False, use_rerank=False),
    "B +hybrid": dict(use_hybrid=True, use_rerank=False),
    "C +rerank": dict(use_hybrid=True, use_rerank=True),
}

consultas_demo = [
    "que dice el articulo 64 del codigo sustantivo del trabajo",   # termino exacto
    "me despidieron sin pagarme la liquidacion, que hago",          # coloquial
    "cual es el plazo para apelar una multa de transito en Argentina",  # fuera de corpus
]

for q in consultas_demo:
    print("=" * 78)
    print("CONSULTA:", q)
    for nombre, banderas in SISTEMAS.items():
        # min_score=None para ver que recupera cada configuracion antes de la
        # valvula de escape.
        crudos = retrieve(q, store, top_k=3, min_score=None, bm25=bm25, **banderas)
        citas = [f"{r.cita} (dense={r.dense_score:.2f})" if r.dense_score is not None else f"{r.cita} (solo BM25)" for r in crudos]
        print(f"  {nombre}: {citas}")

CONSULTA: que dice el articulo 64 del codigo sustantivo del trabajo


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

  C +rerank: ['Decreto 2663 de 1950 (Codigo Sustantivo del Trabajo), Articulo 64 (solo BM25)', 'Constitucion Politica de 1991, Articulo 64 (dense=0.85)', 'Ley 1564 de 2012 (Codigo General del Proceso), Articulo 64 (solo BM25)']
CONSULTA: me despidieron sin pagarme la liquidacion, que hago
  A denso: ['Ley 1564 de 2012 (Codigo General del Proceso), Articulo 563 (dense=0.84)', 'Decreto 2663 de 1950 (Codigo Sustantivo del Trabajo), Articulo 419 (dense=0.83)', 'Ley 1564 de 2012 (Codigo General del Proceso), Articulo 571 (dense=0.83)']
  B +hybrid: ['Ley 1564 de 2012 (Codigo General del Proceso), Articulo 563 (dense=0.84)', 'Ley 1480 de 2011 (Estatuto del Consumidor), Articulo 50 (solo BM25)', 'Decreto 2663 de 1950 (Codigo Sustantivo del Trabajo), Articulo 419 (dense=0.83)']
  C +rerank: ['Decreto 2663 de 1950 (Codigo Sustantivo del Trabajo), Articulo 419 (dense=0.83)', 'Decreto 2663 de 1950 (Codigo Sustantivo del Trabajo), Articulo 263 (dense=0.83)', 'Ley 1564 de 2012 (Codigo General del P

## Fase 3 -- Cargar el generador

Qwen2.5-7B-Instruct, cargado una sola vez y reusado. `USE_LORA=True` genera con
el adaptador LoRA encima.

In [8]:
# Amparo = modelo fine-tuneado de M1 (LoRA) + RAG: es el sistema que se entrega,
# asi que la corrida principal usa el adaptador. False sirve como ablacion
# ("¿el fine-tuning sigue aportando con RAG?"); OJO: en S08 los archivos de
# corridas_abc/ no llevan sufijo; una segunda corrida sobrescribe la primera.
USE_LORA = True
model_bundle = pipeline.load_model(use_lora=USE_LORA)
print("Modelo cargado.")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Modelo cargado.


## Fase 4 -- Corrida A/B/C sobre el eval set (con latencia)

Corre las tres configuraciones sobre `data/eval_set.json` y produce, por cada
una, la lista de registros en formato Ragas (`pipeline.to_eval_record`). Mide la
latencia por consulta de cada configuracion: toda tecnica cobra en tiempo, y ese
costo se reporta junto a la calidad.

Las tres corridas quedan etiquetadas por configuracion (`used_hybrid`/
`used_rerank`) y son comparables: mismo eval set, mismo generador.

In [9]:
import json, time
from tools.evaluation import eval_set as eval_set_mod

registros = eval_set_mod.load_eval_set()
print(f"{len(registros)} registros ({len(eval_set_mod.gold_examples(registros))} gold, "
      f"{len(eval_set_mod.adversarial_examples(registros))} adversariales)")

corridas = {}   # nombre_config -> lista de eval_records
latencias = {}  # nombre_config -> seg/consulta

for nombre, banderas in SISTEMAS.items():
    print(f"\nCorriendo configuracion {nombre} ...")
    records, t0 = [], time.perf_counter()
    for reg in registros:
        consulta = reg["messages"][1]["content"]
        resultado = pipeline.answer_query(
            consulta, store, use_lora=USE_LORA, bm25=bm25,
            model_bundle=model_bundle, **banderas,
        )
        records.append(pipeline.to_eval_record(resultado, reg))
    latencias[nombre] = (time.perf_counter() - t0) / len(registros)
    corridas[nombre] = records
    print(f"  {nombre}: {latencias[nombre]:.2f} seg/consulta")

75 registros (45 gold, 30 adversariales)

Corriendo configuracion A denso ...
  A denso: 5.74 seg/consulta

Corriendo configuracion B +hybrid ...
  B +hybrid: 5.64 seg/consulta

Corriendo configuracion C +rerank ...
  C +rerank: 5.76 seg/consulta


In [10]:
# Persistir las tres corridas en Drive para levantarlas sin re-generar.
destino = pathlib.Path(config.DRIVE_ROOT) / "rag" / "corridas_abc"
destino.mkdir(parents=True, exist_ok=True)

for nombre, records in corridas.items():
    slug = nombre.split()[0].lower()   # a / b / c
    ruta = destino / f"eval_records_config_{slug}.json"
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"  {nombre} -> {ruta}")

print("\nLatencia por consulta:")
for nombre, seg in latencias.items():
    print(f"  {nombre}: {seg:.2f} s")

  A denso -> /content/drive/MyDrive/Colab Notebooks/Amparo/rag/corridas_abc/eval_records_config_a.json
  B +hybrid -> /content/drive/MyDrive/Colab Notebooks/Amparo/rag/corridas_abc/eval_records_config_b.json
  C +rerank -> /content/drive/MyDrive/Colab Notebooks/Amparo/rag/corridas_abc/eval_records_config_c.json

Latencia por consulta:
  A denso: 5.74 s
  B +hybrid: 5.64 s
  C +rerank: 5.76 s


## Fase 5 -- ¿Que busqueda es mejor?

Esta es la pregunta de la S08. Amparo solo puede responder bien si **primero
encuentra la norma correcta**: si la busqueda trae el articulo equivocado (o
nada), el modelo responde mal o inventa, por bueno que sea el prompt. Por eso
aqui se deja fijo todo lo demas (mismo prompt, mismo modelo, mismas preguntas) y
lo **unico que cambia es como se busca**:

| Config | Como busca | En palabras simples |
|---|---|---|
| **A** denso | embeddings e5 (coseno) | busca por **significado**: entiende "me echaron del trabajo" aunque la norma diga "terminacion del contrato", pero confunde numeros ("articulo 64" con el 46 o el 65) |
| **B** + hybrid | denso + BM25, fusion RRF | suma una busqueda por **palabras exactas**: rescata "articulo 64", "Ley 1480", "habeas data", que el denso difumina |
| **C** + rerank | hybrid y un cross-encoder reordena | trae mas candidatos y un segundo modelo **lee cada par pregunta-articulo** para dejar arriba los que de verdad responden y quitar el ruido |

**Que significa "mejor".** No hay una sola nota; cada numero de la tabla mira
una cosa, y todos se calculan sin juez, directo de las respuestas guardadas:

- **Consultas sin contexto:** cuantas veces la busqueda no trajo nada. Menos es
  mejor en los casos gold (en los adversariales, a veces no traer nada es lo
  correcto).
- **Honestidad con las fuentes:** si la respuesta cita solo articulos que la
  busqueda trajo, y la **prudencia** en los adversariales (sin juez, reglas del
  repo).
- **Latencia:** cada tecnica cuesta tiempo; una mejora que no paga su costo no se
  justifica.

La calidad de las respuestas con juez (RAGAS: si la busqueda trae lo necesario y
sin ruido) se mide en `m3_s10_rag_agentico.ipynb`, que es donde esta el juez Groq.

**Antes y despues.** La primera corrida (2026-09-27) destapo tres fallas: la
valvula de escape del prompt nunca se activo, hubo citas inventadas, y el hybrid
descartaba justo lo que BM25 encontraba (por eso B y C traian menos articulos que
A). Se corrigieron en el codigo (`docs/m3_decisiones_rag.md`, seccion 25). La
tabla de abajo compara esa corrida (`corridas_abc_antes/`) con la corrida con las
correcciones (`corridas_abc/`).

### Tabla de comparacion A/B/C

**Sin juez** (se calcula de los archivos `eval_records_config_{a,b,c}.json`, con
`ragas_metrics.tasas_de_escape` y `dspy_prompt.puntaje_de_record`):

| | A antes | A despues | B antes | B despues | C antes | C despues |
|---|---|---|---|---|---|---|
| Consultas sin contexto (gold) | 1 (0) | 1 (0) | 2 (1) | 1 (0) | 5 (4) | **1 (0)** |
| Articulos por consulta | 4.57 | 4.57 | 3.55 | **4.57** | 3.71 | **4.57** |
| Respuestas gold que citan algun articulo | 22 % | 20 % | 10 % | 20 % | 10 % | 12 % |
| Respuestas gold con cita no respaldada | 1 | **0** | 0 | 0 | 0 | 0 |
| Respuestas corregidas por la verificacion de citas | - | 1 (9034) | - | 1 (9102) | - | 0 |
| Prudencia en adversariales | 6/6 | 6/6 | 5/6 | **6/6** | 6/6 | 6/6 |
| Honestidad con las fuentes (0-1) | 0.634 | 0.643 | 0.571 | **0.643** | 0.562 | **0.607** |
| Uso de la frase de escape | 0/56 | 1/56 (9101, por codigo) | 0/56 | 1/56 (9101) | 0/56 | 1/56 (9101) |
| Latencia (s/consulta) | 6.42 | 6.43 | 6.34 | 6.55 | 6.30 | 6.28 |

"Antes": corrida del 2026-09-27 en la manana, sin las correcciones
(`corridas_abc_antes/`). "Despues": corrida del mismo dia con las correcciones de
la seccion 25 de `docs/m3_decisiones_rag.md` (`corridas_abc/`). Ambas con
`USE_LORA = True`, mismas 56 preguntas. Detalle y casos en
`results/m3_s08_busqueda_2026-09-27.md`.

**Que dicen los numeros:**

- **La correccion del piso funciono.** B y C ya no pierden articulos (4.57 por
  consulta, igual que A) y C paso de 4 casos gold sin contexto a 0.
- **La verificacion de citas atrapo las dos citas inventadas** que habian salido
  en la corrida anterior: el "articulo 24" (9034, en A) y el "articulo 2" como el
  de la tutela (9102, en B). En los dos casos el modelo volvio a inventarlas, el
  codigo las rechazo y la respuesta regenerada ya no las cita.
- **El escape por codigo se activo donde debia:** 9101 (California), el unico caso
  sin contexto, responde la frase de escape con la orientacion fija.
- **Sin juez, A y B quedan empatados y C un poco por debajo** en honestidad (C
  cita menos articulos). B recupera algo distinto a A en 36 de 56 preguntas y C en
  49, asi que la diferencia real de calidad entre las tres busquedas la dice
  RAGAS (context recall y precision), en la fase 5b de
  `m3_s10_rag_agentico.ipynb`.
- **La latencia no cambia** (~6.3-6.5 s): la domina la generacion, no la busqueda.

Lectura:
- **B mejor que A** si trae articulos en mas consultas (menos "sin contexto") y
  la honestidad no baja: el corpus tiene vocabulario exacto que el denso
  difuminaba y BM25 lo rescata.
- **C mejor que B** si, con los mismos articulos disponibles, las respuestas se
  apoyan mas en ellos (mas citas respaldadas): el reranker dejo arriba lo que de
  verdad responde.
- **Antes vs despues**: muestra el efecto de las correcciones de la seccion 25
  (sobre todo, que B y C ya no pierdan articulos y que el escape funcione).
- Una tecnica que no mejora ningun numero no ataca un fallo real; se reporta
  junto a su latencia para decidir si su costo se justifica.

---
### Contenido

1. Codigo de las dos tecnicas (`tools/rag/hybrid.py`, `tools/rag/rerank.py`),
   integradas en `retrieve.py`/`pipeline.py` por bandera, con tests que corren
   sin GPU (`tests/rag/`).
2. Las tres corridas A/B/C en formato Ragas y la latencia por configuracion
   (Fase 4).
3. Las decisiones de diseno en `docs/m3_decisiones_rag.md`.

*Amparo · M3 · S08 · RAG avanzado.*

## Manifiesto de la corrida

Guarda junto a los resultados **con que** se produjeron: commit, versiones de
las librerias y huellas del indice, el corpus, el dataset, el eval set y el
adaptador.

Huellas y no fechas: dos corridas pueden decir las dos "2026-10-07" y haber
medido sobre indices distintos. Si declaran el mismo hash, midieron sobre lo
mismo.


In [ ]:
import pathlib

# Manifiesto de la corrida: commit, versiones y huellas del indice, el corpus,
# el dataset, el eval set y el adaptador. Sin esto no se sabe con que salio cada
# cifra -- el adaptador de Drive ya se sobrescribio una vez (6 de octubre) y las
# corridas anteriores a esa fecha quedaron sin forma de reproducirse.
import subprocess

from tools.rag import manifiesto

_hardware = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()

_manifiesto = manifiesto.construir(
    "s08",
    use_lora=USE_LORA,
    n_chunks=len(store.metadata),
    n_eval_set=len(eval_records),
    hardware=_hardware,
)
_destino = pathlib.Path(config.DRIVE_ROOT) / "rag" / "corridas_abc"
print("Manifiesto:", _manifiesto.guardar(_destino))
print(f"  commit {_manifiesto.git_commit[:8]} | indice {_manifiesto.hash_indice} "
      f"| adaptador {_manifiesto.hash_adaptador} | eval set {_manifiesto.hash_eval_set}")
